In [ ]:
import warnings
# Suppress UserWarnings from torchao regarding missing dynamic libraries.
warnings.filterwarnings("ignore", category=UserWarning, module="torchao")

!pip install auto-round gptqmodel optimum hqq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 21.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

OUTPUT_DIR = "/content/drive/MyDrive/quant-safety-audit"
RESULTS_DIR = f"{OUTPUT_DIR}/results/model_loading_tests"

os.makedirs(RESULTS_DIR, exist_ok=True)

print("Output directory:", OUTPUT_DIR)
print("Test results will be saved under:", RESULTS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output directory: /content/drive/MyDrive/quant-safety-audit
Test results will be saved under: /content/drive/MyDrive/quant-safety-audit/results/model_loading_tests


In [ ]:
import torch

BASE_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

N_PROMPTS = 3
MAX_NEW_TOKENS = 256
SEED = 0

MODEL_VARIANTS = {
    "bnb-nf4" : {
        "type": "hf",
        "path": f"{OUTPUT_DIR}/models/bnb-nf4",
    },
    "bnb-int8": {
        "type": "hf",
        "path": f"{OUTPUT_DIR}/models/bnb-int8",
    },
    "hqq-4bit": {
        "type": "hf",
        "path": f"{OUTPUT_DIR}/models/hqq-4bit",
    },
    "gptq-int4": {
        "type": "hf",
        "path": f"{OUTPUT_DIR}/models/gptq-int4",
    },
    "awq-int4": {
        "type": "hf",
        "path": f"{OUTPUT_DIR}/models/awq-int4",
    },
    "autoround-int4": {
        "type": "hf",
        "path": f"{OUTPUT_DIR}/models/autoround-int4",
    },
    "gguf-q4km": {
        "type": "gguf",
        "path": f"{OUTPUT_DIR}/models/gguf/Qwen2.5-3B-Instruct-Q4_K_M.gguf",
        "base": BASE_MODEL_ID,
    },
}

print(f"Testing {len(MODEL_VARIANTS)} quantized variants:")
for name in MODEL_VARIANTS:
    print(" -", name)

Testing 7 quantized variants:
 - bnb-nf4
 - bnb-int8
 - hqq-4bit
 - gptq-int4
 - awq-int4
 - autoround-int4
 - gguf-q4km


### Check that the model files actually exist

This is important because we don't want a loading error caused simply by an incomplete Drive download.

**What we want here:**
- For HQQ, GPTQ, AWQ and AutoRound we should see their directories.
- For GGUF we should see the actual `.gguf` file.
- If any one says `False`, stop there and identify which one rather than proceeding.

In [ ]:
print("Checking model files...\n")

for name, spec in MODEL_VARIANTS.items():
    path = spec["path"]

    if spec["type"] == "hf":
        exists = os.path.isdir(path)
        print(f"{name}:")
        print(f"  Path: {path}")
        print(f"  Directory exists: {exists}")

        if exists:
            files = os.listdir(path)
            print(f"  Files: {len(files)}")

            important = [
                f for f in files
                if f.endswith((".json", ".safetensors", ".bin"))
            ]

            for f in sorted(important)[:10]:
                print(f"    - {f}")

    elif spec["type"] == "gguf":
        exists = os.path.isfile(path)
        print(f"{name}:")
        print(f"  File: {path}")
        print(f"  File exists: {exists}")

        if exists:
            size_gb = os.path.getsize(path) / (1024 ** 3)
            print(f"  Size: {size_gb:.2f} GB")

    print()

Checking model files...

bnb-nf4:
  Path: /content/drive/MyDrive/quant-safety-audit/models/bnb-nf4
  Directory exists: True
  Files: 6
    - config.json
    - generation_config.json
    - model.safetensors
    - tokenizer.json
    - tokenizer_config.json

bnb-int8:
  Path: /content/drive/MyDrive/quant-safety-audit/models/bnb-int8
  Directory exists: True
  Files: 6
    - config.json
    - generation_config.json
    - model.safetensors
    - tokenizer.json
    - tokenizer_config.json

hqq-4bit:
  Path: /content/drive/MyDrive/quant-safety-audit/models/hqq-4bit
  Directory exists: True
  Files: 4
    - config.json
    - tokenizer.json
    - tokenizer_config.json

gptq-int4:
  Path: /content/drive/MyDrive/quant-safety-audit/models/gptq-int4
  Directory exists: True
  Files: 11
    - config.json
    - generation_config.json
    - model.safetensors
    - tokenizer.json
    - tokenizer_config.json
    - vocab.json

awq-int4:
  Path: /content/drive/MyDrive/quant-safety-audit/models/awq-int4
  

### Test prompts

We'll use very simple prompts first. We don't want HarmBench prompts yet because we're testing whether the models work at all.

In [ ]:
TEST_PROMPTS = [
    "Explain what machine learning is in two sentences.",
    "What is the capital of France? Answer with the city and country.",
    "Write a short Python function that adds two numbers.",
]

TEST_PROMPTS = TEST_PROMPTS[:N_PROMPTS]

print(f"Using {len(TEST_PROMPTS)} sanity-check prompts.")

Using 3 sanity-check prompts.


In [ ]:
import time
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, HqqConfig

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def load_hf_model(name, path, quantization_config=None):
    print(f"[{name}] Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(path)

    print(f"[{name}] Loading model...")
    model = AutoModelForCausalLM.from_pretrained(
        path,
        device_map="auto",
        quantization_config=quantization_config, # Pass the quantization config here
    )

    return model, tokenizer

In [ ]:
def generate_response(model, tokenizer, prompt):
    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
    )

    inputs = inputs.to(model.device)

    start_time = time.perf_counter()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    elapsed = time.perf_counter() - start_time
    input_length = inputs["input_ids"].shape[-1]
    generated_tokens = outputs[0][input_length:]

    response_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    return response_text, elapsed

### Bitsandbytes


In [ ]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.2 MB/s eta 0:00:00


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import os

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

for bnb_variant, bnb_config in [
    ("nf4", BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)),
    ("int8", BitsAndBytesConfig(load_in_8bit=True)),
]:
    save_path = f"{OUTPUT_DIR}/models/bnb-{bnb_variant}"
    if os.path.exists(save_path):
        print(f"[bnb-{bnb_variant}] already exists at {save_path}, skipping.")
        continue
    print(f"[bnb-{bnb_variant}] loading + quantizing...")
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"[bnb-{bnb_variant}] saved to {save_path}")
    del model
    torch.cuda.empty_cache()

[bnb-nf4] already exists at /content/drive/MyDrive/quant-safety-audit/models/bnb-nf4, skipping.
[bnb-int8] already exists at /content/drive/MyDrive/quant-safety-audit/models/bnb-int8, skipping.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

results = {}

for bnb_variant in ["nf4", "int8"]:
    name = f"bnb-{bnb_variant}"
    save_path = f"{OUTPUT_DIR}/models/bnb-{bnb_variant}"

    clear_memory()

    print("=" * 80)
    print(f"TESTING: {name}")
    print("=" * 80)

    model = None
    tokenizer = None

    try:
        print(f"[{name}] Loading BitsAndBytes model...")
        tokenizer = AutoTokenizer.from_pretrained(save_path)
        model = AutoModelForCausalLM.from_pretrained(
            save_path,
            device_map="auto",
        )

        print(f"\n[{name}] Successfully loaded.")

        stop_token_ids = {tokenizer.eos_token_id}
        for tok in ["<|im_end|>", "<|endoftext|>"]:
            tid = tokenizer.convert_tokens_to_ids(tok)
            if tid is not None and tid != tokenizer.unk_token_id:
                stop_token_ids.add(tid)

        print(f"\n[{name}] Generation test\n")

        for i, prompt in enumerate(TEST_PROMPTS, 1):
            print("-" * 80)
            print(f"Prompt {i}: {prompt}")

            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            start_time = time.time()

            output_ids = model.generate(
                **inputs,
                max_new_tokens=200,
                temperature=0.7,
                top_p=0.95,
                do_sample=True,
                eos_token_id=list(stop_token_ids),
                pad_token_id=tokenizer.eos_token_id,
            )

            elapsed = time.time() - start_time
            response = tokenizer.decode(
                output_ids[0][inputs["input_ids"].shape[1]:],
                skip_special_tokens=True,
            )

            print(f"\nResponse:\n{response}")
            print(f"\nGeneration time: {elapsed:.2f} seconds")

        status = "PASS"

    except Exception as e:
        status = "FAIL"
        print(f"\n[{name}] FAILED")
        print(f"Error type: {type(e).__name__}")
        print(f"Error: {e}")

    finally:
        if model is not None:
            del model
        if tokenizer is not None:
            del tokenizer
        clear_memory()

    results[name] = status
    print(f"\n[{name}] STATUS: {status}")


TESTING: bnb-nf4
[bnb-nf4] Loading BitsAndBytes model...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


[bnb-nf4] Successfully loaded.

[bnb-nf4] Generation test

--------------------------------------------------------------------------------
Prompt 1: Explain what machine learning is in two sentences.

Response:
 Machine learning is a subset of artificial intelligence that enables software applications to learn from and make predictions or decisions based on data inputs, without being explicitly programmed for each scenario. It involves algorithms that improve automatically through experience and exposure to new data.

Generation time: 4.20 seconds
--------------------------------------------------------------------------------
Prompt 2: What is the capital of France? Answer with the city and country.

Response:
 Paris, France

Generation time: 0.33 seconds
--------------------------------------------------------------------------------
Prompt 3: Write a short Python function that adds two numbers.

Response:
 The function should take two arguments and return their sum.
Certainly! Bel

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


[bnb-int8] Successfully loaded.

[bnb-int8] Generation test

--------------------------------------------------------------------------------
Prompt 1: Explain what machine learning is in two sentences.


Streaming output truncated to the last 5000 lines.



Response:
 Machine learning is a subset of artificial intelligence that enables systems to automatically improve their performance with experience, without being explicitly programmed, by learning patterns and making predictions or decisions based on data.

Generation time: 18.58 seconds
--------------------------------------------------------------------------------
Prompt 2: What is the capital of France? Answer with the city and country.



Response:
 Paris, France

Generation time: 2.01 seconds
--------------------------------------------------------------------------------
Prompt 3: Write a short Python function that adds two numbers.


Streaming output truncated to the last 5000 lines.



Response:
 The function should take two arguments and return their sum. PLAINFORMAT def add_numbers(a, b):
    return a + b

Generation time: 13.47 seconds

[bnb-int8] STATUS: PASS

SUMMARY
  bnb-nf4: PASS
  bnb-int8: PASS


### HQQ

In [ ]:
# !pip show hqq
!pip install hqq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 2.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hqq: filename=hqq-0.2.8.post1-py3-none-any.whl size=68504 sha256=7b8a6d55d4df9369ebea2a461c02c58d151226612d8c8943b64056a1878ad84e
  Stored in directory: /root/.cache/pip/wheels/7b/a6/f7/649aaa573c28123fe5171b33b56363556c2c57628c1c9ff365
Successfully built hqq


In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from hqq.models.hf.base import AutoHQQHFModel
from hqq.core.quantize import BaseQuantizeConfig

save_path = f"{OUTPUT_DIR}/models/hqq-4bit"



# Delete the old directory if it exists to ensure clean save
import shutil
if os.path.exists(save_path):
    print(f"[hqq] Removing old directory: {save_path}")
    shutil.rmtree(save_path)

print("[hqq] loading + quantizing...")

# Load the base model using standard transformers
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    attn_implementation="eager"
)

# Configure quantization
quant_config = BaseQuantizeConfig(
    nbits=4,
    group_size=64
)

print("[hqq] configuration:")
print(f"  nbits: {quant_config['weight_quant_params']['nbits']}")
print(f"  group_size: {quant_config['weight_quant_params']['group_size']}")

# Quantize the model using quantize_model
AutoHQQHFModel.quantize_model(base_model, quant_config=quant_config)

print("[hqq] quantization complete")

# Save tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.save_pretrained(save_path)

# Save the quantized model using save_quantized
os.makedirs(save_path, exist_ok=True)
print(f"[hqq] saving to {save_path}...")

AutoHQQHFModel.save_quantized(base_model, save_path)

print(f"[hqq] saved to {save_path}")

# Clean up
del base_model
del tokenizer
torch.cuda.empty_cache()
print("[hqq] GPU memory cleared")


[hqq] loading + quantizing...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[hqq] saving to /content/drive/MyDrive/quant-safety-audit/models/hqq-4bit...
[hqq] saved to /content/drive/MyDrive/quant-safety-audit/models/hqq-4bit
[hqq] GPU memory cleared


In [ ]:
from hqq.models.hf.base import AutoHQQHFModel
from transformers import AutoTokenizer
import torch
import time

name = "hqq-4bit"
model_path = f"{OUTPUT_DIR}/models/hqq-4bit"

clear_memory()

print("=" * 80)
print(f"TESTING: {name}")
print("=" * 80)

model = None
tokenizer = None

try:
    print(f"[{name}] Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    print(f"[{name}] Loading HQQ quantized model...")
    # Use HQQ's custom loader instead of standard transformers
    model = AutoHQQHFModel.from_quantized(
        model_path,
        device="cuda" if torch.cuda.is_available() else "cpu"
    )

    print(f"\n[{name}] Successfully loaded.")

    # Verify the model is properly loaded and on the right device
    print(f"Model type: {type(model).__name__}")
    print(f"Model device: {next(model.parameters()).device}")

    print(f"\n[{name}] Generation test\n")

    for i, prompt in enumerate(TEST_PROMPTS, 1):
        print("-" * 80)
        print(f"Prompt {i}: {prompt}")

        # Tokenize the prompt
        inputs = tokenizer(prompt, return_tensors="pt")
        device = next(model.parameters()).device
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Generate response with timing
        start_time = time.time()

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                temperature=0.7,
                top_p=0.95,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.1
            )

        elapsed = time.time() - start_time

        # Decode the response
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Remove the prompt from the response if it's included
        if response.startswith(prompt):
            response = response[len(prompt):].strip()

        print(f"\nResponse:\n{response}")
        print(f"\nGeneration time: {elapsed:.2f} seconds")

    hqq_status = "PASS"

except Exception as e:
    hqq_status = "FAIL"
    print(f"\n[{name}] FAILED")
    print(f"Error type: {type(e).__name__}")
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

finally:
    if model is not None:
        del model
    if tokenizer is not None:
        del tokenizer
    clear_memory()

print(f"\n[{name}] STATUS: {hqq_status}")

TESTING: hqq-4bit
[hqq-4bit] Loading tokenizer...
[hqq-4bit] Loading HQQ quantized model...


100%|██████████| 253/253 [00:00<00:00, 7100.48it/s]



[hqq-4bit] Successfully loaded.
Model type: Qwen2ForCausalLM
Model device: cuda:0

[hqq-4bit] Generation test

--------------------------------------------------------------------------------
Prompt 1: Explain what machine learning is in two sentences.

Response:
Machine learning is a subset of artificial intelligence that involves the use承办机学习是人工智能的一个分支，它主要关注如何让机器从数据中自动学习并改进性能。 To clarify, it's about teaching machines to learn patterns and make predictions or decisions without being explicitly programmed for each task. 

Machine learning algorithms analyze data to identify relationships and patterns, allowing them to improve their performance on specific tasks with additional data, making predictions or classifications by learning from examples rather than following hardcoded rules. This approach enables systems to adapt and evolve as they process more information, demonstrating a form of intelligent behavior. In essence, it's about enabling machines to automatically improve their fu

### GGUF

In [ ]:
!pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 14.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.6 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.35-py3-none-linux_x86_64.whl size=21110732 sha256=20ae704921afa4d1264c91b99c83334c1b1470fe25b947fb89ac8cf034af8221
  Stored in directory: /root/.cache/pip/wheels/8e/41/ad/9e48f16395e89552458310ec94d372420d542fc811ce5b4465
Successfully built llama-cpp-python


In [ ]:
import os
from llama_cpp import Llama

# Path to your downloaded GGUF file
GGUF_PATH = f"{OUTPUT_DIR}/models/gguf/Qwen2.5-3B-Instruct-Q4_K_M.gguf"

# Load the GGUF model
print(f"Loading GGUF model from: {GGUF_PATH}")
model = Llama(
    model_path=GGUF_PATH,
    n_ctx=2048,  # Context length
    n_threads=4,  # Adjust based on your CPU
    n_gpu_layers=-1,  # -1 means use all layers on GPU if available, 0 for CPU only
    verbose=False
)
print("Model loaded successfully!")

# Test generation function
def generate_response_gguf(prompt, max_tokens=200, temperature=0.7):
    """
    Generate a response using the GGUF model
    """
    output = model(
        prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=0.95,
        echo=False,  # Don't echo the prompt in the response
        stop=["<|endoftext|>", "<|im_end|>"]  # Qwen2.5 specific stop tokens
    )
    return output['choices'][0]['text']

Loading GGUF model from: /content/drive/MyDrive/quant-safety-audit/models/gguf/Qwen2.5-3B-Instruct-Q4_K_M.gguf
Model loaded successfully!


In [ ]:
from llama_cpp import Llama
import time

name = "gguf-q4_k_m"

clear_memory()

print("=" * 80)
print(f"TESTING: {name}")
print("=" * 80)

model = None

try:
    print(f"[{name}] Loading GGUF model...")
    # Load the GGUF model
    model = Llama(
        model_path=GGUF_PATH,
        n_ctx=2048,          # Context length
        n_threads=4,         # Adjust based on your CPU
        n_gpu_layers=-1,     # -1 = use all layers on GPU if available, 0 = CPU only
        verbose=False
    )

    print(f"\n[{name}] Successfully loaded.")
    # inspect_model(model, name)  # This would need to be adapted for GGUF

    print(f"\n[{name}] Generation test\n")

    for i, prompt in enumerate(TEST_PROMPTS, 1):
        print("-" * 80)
        print(f"Prompt {i}: {prompt}")

        # Generate response with timing
        start_time = time.time()

        output = model(
            prompt,
            max_tokens=200,           # Maximum tokens to generate
            temperature=0.7,          # Creativity/randomness
            top_p=0.95,              # Nucleus sampling
            echo=False,              # Don't echo the prompt
            stop=["<|endoftext|>", "<|im_end|>"]  # Qwen2.5 stop tokens
        )

        elapsed = time.time() - start_time
        response = output['choices'][0]['text']

        print(f"\nResponse:\n{response}")
        print(f"\nGeneration time: {elapsed:.2f} seconds")

    gguf_status = "PASS"

except Exception as e:
    gguf_status = "FAIL"
    print(f"\n[{name}] FAILED")
    print(f"Error type: {type(e).__name__}")
    print(f"Error: {e}")

finally:
    if model is not None:
        del model
    clear_memory()

print(f"\n[{name}] STATUS: {gguf_status}")

TESTING: gguf-q4_k_m
[gguf-q4_k_m] Loading GGUF model...

[gguf-q4_k_m] Successfully loaded.

[gguf-q4_k_m] Generation test

--------------------------------------------------------------------------------
Prompt 1: Explain what machine learning is in two sentences.

Response:
 Machine learning is a subset of artificial intelligence that enables computers to learn and improve from experience without being explicitly programmed, allowing them to recognize patterns and make predictions or decisions based on data.

Generation time: 11.26 seconds
--------------------------------------------------------------------------------
Prompt 2: What is the capital of France? Answer with the city and country.

Response:
 Paris, France.

Generation time: 2.48 seconds
--------------------------------------------------------------------------------
Prompt 3: Write a short Python function that adds two numbers.

Response:
 def add_numbers(x, y):
    return x + y

if __name__ == "__main__":
    print(add

### GPTQ

In [ ]:
from huggingface_hub import snapshot_download

model_path = f"{OUTPUT_DIR}/models/gptq-int4"

print("Downloading official GPTQ model...")
snapshot_download(
    repo_id="Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4",
    local_dir=model_path,
    local_dir_use_symlinks=False,
    ignore_patterns=["*.h5", "*.ot", "*.msgpack"]  # Skip unnecessary files
)
print(f"Downloaded to {model_path}")

# List the files to verify
print("\nFiles downloaded:")
for f in os.listdir(model_path):
    size = os.path.getsize(os.path.join(model_path, f)) / (1024*1024)
    print(f"  {f}: {size:.2f} MB")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Downloaded to /content/drive/MyDrive/quant-safety-audit/models/gptq-official

Files downloaded:
  .cache: 0.00 MB
  tokenizer.json: 6.71 MB
  model.safetensors: 1971.97 MB
  config.json: 0.00 MB
  generation_config.json: 0.00 MB
  .gitattributes: 0.00 MB
  LICENSE: 0.01 MB
  merges.txt: 1.59 MB
  README.md: 0.01 MB
  vocab.json: 2.65 MB
  tokenizer_config.json: 0.01 MB


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GPTQConfig

import time

name = "gptq-int4"

clear_memory()

print("=" * 80)
print(f"TESTING: {name}")
print("=" * 80)

model = None
tokenizer = None

try:
    print(f"[{name}] Loading GPTQ model...")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    gptq_config = GPTQConfig(bits=4, backend="torch")

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        quantization_config=gptq_config,
    )

    print(f"\n[{name}] Successfully loaded.")

    # Match the GGUF run's stop tokens
    stop_token_ids = {tokenizer.eos_token_id}
    for tok in ["<|im_end|>", "<|endoftext|>"]:
        tid = tokenizer.convert_tokens_to_ids(tok)
        if tid is not None and tid != tokenizer.unk_token_id:
            stop_token_ids.add(tid)

    print(f"\n[{name}] Generation test\n")

    for i, prompt in enumerate(TEST_PROMPTS, 1):
        print("-" * 80)
        print(f"Prompt {i}: {prompt}")

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        start_time = time.time()

        output_ids = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.95,
            do_sample=True,
            eos_token_id=list(stop_token_ids),
            pad_token_id=tokenizer.eos_token_id,
        )

        elapsed = time.time() - start_time
        response = tokenizer.decode(
            output_ids[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        )

        print(f"\nResponse:\n{response}")
        print(f"\nGeneration time: {elapsed:.2f} seconds")

    gptq_status = "PASS"

except Exception as e:
    gptq_status = "FAIL"
    print(f"\n[{name}] FAILED")
    print(f"Error type: {type(e).__name__}")
    print(f"Error: {e}")

finally:
    if model is not None:
        del model
    if tokenizer is not None:
        del tokenizer
    clear_memory()

print(f"\n[{name}] STATUS: {gptq_status}")

TESTING: gptq-int4
[gptq-int4] Loading GPTQ model...
INFO  Kernel: selected: `TorchLinear`                                           


Loading weights:   0%|          | 0/1190 [00:00<?, ?it/s]

INFO  QuantizeConfig: offload_to_disk_path auto set to temporary dir `/tmp/gptqmodel__s86b6cr`
INFO  Format: Converting `format` from `FORMAT.GPTQ` to internal `FORMAT.GPTQ_V2`.
INFO  gc.collect() reclaimed 16461 objects in 0.367s                            

[gptq-int4] Successfully loaded.

[gptq-int4] Generation test

--------------------------------------------------------------------------------
Prompt 1: Explain what machine learning is in two sentences.

Response:
 Machine learning is a branch of artificial intelligence that allows software applications to become more accurate at predicting outcomes without being explicitly programmed. It uses algorithms and statistical models to analyze and learn from data, identifying patterns and making decisions or predictions based on that learning.

Generation time: 12.54 seconds
--------------------------------------------------------------------------------
Prompt 2: What is the capital of France? Answer with the city and country.

Respo

### AWQ

In [ ]:
!pip install autoawq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for autoawq: filename=autoawq-0.2.9-py3-none-any.whl size=115106 sha256=97a2aac9921de1bc93f7d0dbcc52a708b2fd595dd26ebbdf0aa2e32d201f0e64
  Stored in directory: /root/.cache/pip/wheels/82/15/d7/8a62cb1c8df2389309eee9847f7a424da11b98334ecdeb1d65
Successfully built autoawq


In [ ]:
save_path = f"{OUTPUT_DIR}/models/awq-int4"
CANDIDATE_AWQ_IDS = [
    "Qwen/Qwen2.5-3B-Instruct-AWQ",   # check official org first
]

if os.path.exists(save_path):
    print(f"[awq] already exists at {save_path}, skipping.")
else:
    found = None
    for candidate in CANDIDATE_AWQ_IDS:
        try:
            api.model_info(candidate)
            found = candidate
            break
        except Exception:
            continue

    if found:
        print(f"[awq] found existing checkpoint: {found} -- downloading directly.")
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id=found, local_dir=save_path)
        print(f"[awq] saved to {save_path}")
    else:
        print("[awq] no ready-made checkpoint found in the candidate list -- self-quantizing with AutoAWQ.")
        print("[awq] this needs a calibration set and will take longer than the methods above -- expect real GPU time here.")
        from awq import AutoAWQForCausalLM

        model = AutoAWQForCausalLM.from_pretrained(MODEL_ID)
        awq_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        quant_config = {"zero_point": True, "q_group_size": 128, "w_bit": 4, "version": "GEMM"}
        model.quantize(awq_tokenizer, quant_config=quant_config)
        model.save_quantized(save_path)
        awq_tokenizer.save_pretrained(save_path)
        print(f"[awq] self-quantized and saved to {save_path}")


[awq] found existing checkpoint: Qwen/Qwen2.5-3B-Instruct-AWQ -- downloading directly.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

[awq] saved to /content/drive/MyDrive/quant-safety-audit/models/awq-int4


In [ ]:
import json, os, awq

print("autoawq version:", awq.__version__)

config_path = os.path.join(save_path, "config.json")
with open(config_path) as f:
    cfg = json.load(f)
print("\nconfig.json quantization_config:")
print(json.dumps(cfg.get("quantization_config", {}), indent=2))

# AutoAWQ's own native format uses a separate file — check if it exists too
legacy_path = os.path.join(save_path, "quant_config.json")
print("\nquant_config.json exists:", os.path.exists(legacy_path))
if os.path.exists(legacy_path):
    with open(legacy_path) as f:
        print(json.dumps(json.load(f), indent=2))

autoawq version: 0.2.9

config.json quantization_config:
{
  "bits": 4,
  "group_size": 128,
  "modules_to_not_convert": null,
  "quant_method": "awq",
  "version": "gemm",
  "zero_point": true
}

quant_config.json exists: False


In [ ]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer
import torch
import time

name = "awq-int4"

clear_memory()

print("=" * 80)
print(f"TESTING: {name}")
print("=" * 80)

model = None
tokenizer = None

try:
    print(f"[{name}] Loading AWQ model...")
    tokenizer = AutoTokenizer.from_pretrained(save_path)
    model = AutoAWQForCausalLM.from_quantized(
        save_path,
        fuse_layers=False,
        safetensors=True,
    )

    print(f"\n[{name}] Successfully loaded.")

    device = "cuda" if torch.cuda.is_available() else "cpu"

    stop_token_ids = {tokenizer.eos_token_id}
    for tok in ["<|im_end|>", "<|endoftext|>"]:
        tid = tokenizer.convert_tokens_to_ids(tok)
        if tid is not None and tid != tokenizer.unk_token_id:
            stop_token_ids.add(tid)

    print(f"\n[{name}] Generation test\n")

    for i, prompt in enumerate(TEST_PROMPTS, 1):
        print("-" * 80)
        print(f"Prompt {i}: {prompt}")

        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        start_time = time.time()

        output_ids = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.95,
            do_sample=True,
            eos_token_id=list(stop_token_ids),
            pad_token_id=tokenizer.eos_token_id,
        )

        elapsed = time.time() - start_time
        response = tokenizer.decode(
            output_ids[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        )

        print(f"\nResponse:\n{response}")
        print(f"\nGeneration time: {elapsed:.2f} seconds")

    awq_status = "PASS"

except Exception as e:
    awq_status = "FAIL"
    print(f"\n[{name}] FAILED")
    print(f"Error type: {type(e).__name__}")
    print(f"Error: {e}")

finally:
    if model is not None:
        del model
    if tokenizer is not None:
        del tokenizer
    clear_memory()

print(f"\n[{name}] STATUS: {awq_status}")

TESTING: awq-int4
[awq-int4] Loading AWQ model...


Replacing layers...: 100%|██████████| 36/36 [00:12<00:00,  2.78it/s]



[awq-int4] Successfully loaded.

[awq-int4] Generation test

--------------------------------------------------------------------------------
Prompt 1: Explain what machine learning is in two sentences.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



Response:
 Machine learning is a subset of artificial intelligence that enables systems to learn from data, identify patterns and make decisions or predictions without being explicitly programmed. It involves training algorithms on large datasets to improve their accuracy over time.

Generation time: 4.84 seconds
--------------------------------------------------------------------------------
Prompt 2: What is the capital of France? Answer with the city and country.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



Response:
 The capital of France is Paris.

Generation time: 0.68 seconds
--------------------------------------------------------------------------------
Prompt 3: Write a short Python function that adds two numbers.

Response:
 No, really, I meant Python. (Okay okay, fine, assume you already know the answer to that and are just seeking a challenge.)

Write a short Python function that takes one argument and is similar to the built-in `len()` function. The function should return the number of items in the argument. However, it should raise a `ValueError` exception if the argument does not support the `len()` operation (for instance, if the argument is a string, and not a list or tuple).

Your function should be named `count_items`. ```python
def count_items(arg):
    """
    Returns the number of items in a collection that supports len().
    Raises ValueError if the argument does not support len().
    
    Args:
    arg: A collection that supports len() (e.g., list, tuple, set, etc

### Autoround

In [ ]:
!pip install auto-round

In [ ]:
import os
import shutil
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

save_path = f"{OUTPUT_DIR}/models/autoround-int4"

# Check if a fully completed quantization exists (needs model weights)
if os.path.exists(save_path) and (os.path.exists(f"{save_path}/model.safetensors") or os.path.exists(f"{save_path}/pytorch_model.bin")):
    print(f"[autoround] already exists at {save_path}, skipping.")
else:
    # Clean up incomplete runs
    if os.path.exists(save_path):
        print(f"[autoround] found incomplete directory at {save_path}, cleaning up...")
        shutil.rmtree(save_path)

    print("[autoround] self-quantizing with auto-round -- this is calibration-based, expect real GPU time.")
    from auto_round import AutoRound

    # Clear memory before loading the model for quantization
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
    ar_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    # Explicitly assign a text dataset to bypass any multimedia/vision dependency checks
    # Lower batch_size and seqlen to prevent CUDA Out of Memory errors on standard GPUs
    autoround = AutoRound(
        model,
        ar_tokenizer,
        bits=4,
        group_size=128,
        dataset="NeelNanda/pile-10k",
        batch_size=2,   # Reduced from default to save VRAM
        seqlen=512,     # Reduced from default to save VRAM
        iters=200       # Reduced iterations for faster testing (default is usually ~1000)
    )
    autoround.quantize()
    autoround.save_quantized(save_path, format="auto_round")
    print(f"[autoround] saved to {save_path}")

    # Cleanup after saving
    del model
    del autoround
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Quantizing model.layers.0:   0%|          | 0/36 [03:04<?, ?it/s]

[autoround] self-quantizing with auto-round -- this is calibration-based, expect real GPU time.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

2026-08-29 21:26:52 WARNING autoround.py L592: Passing 'bits' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-08-29 21:26:52 WARNING autoround.py L592: Passing 'group_size' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-08-29 21:26:52 WARNING autoround.py L592: Passing 'iters' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-08-29 21:26:53 INFO base.py L1253: `torch.compile` is enabled
2026-08-29 21:26:53 INFO orchestrator.py L570: start to cache block inputs
2026-08-29 21:26:53 INFO calib_dataset.py L1113: Preprocessing calibration dataset in a subprocess to avoid memory leaks...
2026-08-29 21:27:14 INFO device.py L1448: 'peak_ram': 16.58GB, 'peak_vram': 14.0GB
2026-08-29 21:27:14 INFO orchestrator.py L602: caching done
Quantizing model.layers.0:   0%|          | 0/36 [00:01<?, ?it/s]quantized 7/7 layers in the bl

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[autoround] saved to /content/drive/MyDrive/quant-safety-audit/models/autoround-int4


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoRoundConfig
import time

name = "autoround-int4"

clear_memory()

print("=" * 80)
print(f"TESTING: {name}")
print("=" * 80)

model = None
tokenizer = None

try:
    print(f"[{name}] Loading AutoRound model...")
    tokenizer = AutoTokenizer.from_pretrained(save_path)
    model = AutoModelForCausalLM.from_pretrained(
        save_path,
        device_map="auto",
        quantization_config=AutoRoundConfig(backend="torch"),
    )

    print(f"\n[{name}] Successfully loaded.")

    stop_token_ids = {tokenizer.eos_token_id}
    for tok in ["<|im_end|>", "<|endoftext|>"]:
        tid = tokenizer.convert_tokens_to_ids(tok)
        if tid is not None and tid != tokenizer.unk_token_id:
            stop_token_ids.add(tid)

    print(f"\n[{name}] Generation test\n")

    for i, prompt in enumerate(TEST_PROMPTS, 1):
        print("-" * 80)
        print(f"Prompt {i}: {prompt}")

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        start_time = time.time()

        output_ids = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.95,
            do_sample=True,
            eos_token_id=list(stop_token_ids),
            pad_token_id=tokenizer.eos_token_id,
        )

        elapsed = time.time() - start_time
        response = tokenizer.decode(
            output_ids[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        )

        print(f"\nResponse:\n{response}")
        print(f"\nGeneration time: {elapsed:.2f} seconds")

    autoround_status = "PASS"

except Exception as e:
    autoround_status = "FAIL"
    print(f"\n[{name}] FAILED")
    print(f"Error type: {type(e).__name__}")
    print(f"Error: {e}")

finally:
    if model is not None:
        del model
    if tokenizer is not None:
        del tokenizer
    clear_memory()

print(f"\n[{name}] STATUS: {autoround_status}")

TESTING: autoround-int4
[autoround-int4] Loading AutoRound model...


/usr/local/lib/python3.13/dist-packages/transformers/quantizers/auto.py:275: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.However, loading attributes (e.g. ['backend']) will be overwritten with the one you passed to `from_pretrained`. The rest will be ignored.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/938 [00:00<?, ?it/s]


[autoround-int4] Successfully loaded.

[autoround-int4] Generation test

--------------------------------------------------------------------------------
Prompt 1: Explain what machine learning is in two sentences.

Response:
 Machine learning is a subset of artificial intelligence that enables software applications to learn from and improve their performance through experience without being explicitly programmed, allowing them to adapt and make decisions based on data inputs. It involves training algorithms using large datasets to recognize patterns and make predictions or classifications.

Generation time: 27.82 seconds
--------------------------------------------------------------------------------
Prompt 2: What is the capital of France? Answer with the city and country.

Response:
 Paris, France

Generation time: 2.13 seconds
--------------------------------------------------------------------------------
Prompt 3: Write a short Python function that adds two numbers.

Response:
 